# 01 · Recolección y calidad del dato

**Hotel Booking Demand · Machine Learning · MUCSI 2026/27 · Universidad de Deusto**

**Objetivo del proyecto:** cuando entra una reserva, predecir **si se cancelará o no se presentará** (`is_canceled`) y
**cuánto pagará por noche** (`adr`).

> **TL;DR**
>
> | Pregunta | Respuesta |
> |---|---|
> | ¿El fichero es el correcto? | ✅ Reservas y cancelaciones por hotel coinciden con el paper |
> | ¿Hay duplicados? | ⚠️ 27 % de copias exactas |
> | ¿Qué columnas no se pueden usar? | 6 prohibidas (se conocen después de reservar) |
> | ¿Hay filas imposibles? | 182 a cuarentena · 715 de 0 noches fuera de la regresión |
> | ¿Los datos son los del momento de la reserva? | ⚠️ No: son los del día antes de la llegada (paper) |
> | ¿Pasa al notebook 02? | **Sí, con condiciones** |

## 0 · Preparar el entorno

In [1]:
import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", None)

# Project root = the folder that contains requirements.txt (works from notebooks/ or from the root)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").exists())
RAW_PATH = ROOT / "data" / "raw" / "hotels.csv"
OUT_DIR = ROOT / "outputs" / "01-data-collection-and-quality"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def check(condition, ok, fail):
    """Print ✅ if the condition holds and ⚠️ otherwise."""
    print(f"✅ {ok}" if bool(condition) else f"⚠️ {fail}")


def pct(mask):
    """Share of True values, in %."""
    return round(mask.mean() * 100, 1)


def sha256_file(path):
    """File fingerprint: if a single byte changes, the hash changes."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

## 1 · ¿El fichero es el correcto?

Lo comparamos con lo que publicaron los autores: **reservas y cancelaciones por hotel** (tablas 3 y 6 del
paper). El paper no publica ningún hash: el **hash** solo fija la copia que usamos (TidyTuesday) para detectar
si cambia. El paper describe 31 variables por hotel; la copia une los dos hoteles y añade la columna `hotel`.

Fuente: Antonio, de Almeida y Nunes (2019), *Data in Brief* · [DOI](https://doi.org/10.1016/j.dib.2018.11.126) · CC BY 4.0.

In [2]:
SOURCE = {
    "name": "hotel-booking-demand",
    "doi": "10.1016/j.dib.2018.11.126",
    "license": "CC BY 4.0",
    "url": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv",
    "expected_sha256": "7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06",
}
PAPER = pd.DataFrame({  # Tables 3 and 6 of the paper
    "Resort Hotel": {"bookings": 40_060, "canceled": 11_122},
    "City Hotel": {"bookings": 79_330, "canceled": 33_102},
}).T

if not RAW_PATH.exists():  # downloaded only if missing
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(SOURCE["url"], RAW_PATH)

raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)  # all as text: nothing is interpreted yet
raw_sha256 = sha256_file(RAW_PATH)

per_hotel = pd.DataFrame({
    "bookings": raw["hotel"].value_counts(),
    "canceled": raw.loc[raw["is_canceled"] == "1", "hotel"].value_counts(),
}).reindex(PAPER.index)
paper_counts_ok = bool((per_hotel.to_numpy() == PAPER.to_numpy()).all())

check(paper_counts_ok, "Bookings and cancellations per hotel match the paper.", "Counts differ from the paper.")
check(raw_sha256 == SOURCE["expected_sha256"], "Hash matches: same copy as always.", "Hash mismatch: the copy changed.")
check(raw.shape == (119_390, 32), "119,390 rows × 32 columns (the paper's 31 + hotel).", f"Unexpected shape: {raw.shape}")
per_hotel.join(PAPER.add_suffix(" (paper)"))

✅ Bookings and cancellations per hotel match the paper.
✅ Hash matches: same copy as always.
✅ 119,390 rows × 32 columns (the paper's 31 + hotel).


,bookings,canceled,bookings (paper),canceled (paper)
Resort Hotel,40060,11122,40060,11122
City Hotel,79330,33102,79330,33102


## 2 · ¿Qué es cada columna?

Cada fila es **una reserva**. El `role` dice para qué sirve cada columna:

- `feature` → posible entrada del modelo.
- `target_cls` / `target_reg` → lo que predecimos.
- `forbidden` → está en el fichero, pero se rellena después de reservar: **nunca** puede ser entrada (§4).
- `outcome` → el desenlace de la reserva: **nunca** puede ser entrada.

In [3]:
DATA_DICTIONARY = pd.DataFrame([
    ("hotel",                          "categorical", "feature"),
    ("is_canceled",                    "binary",      "target_cls"),
    ("lead_time",                      "count_days",  "feature"),
    ("arrival_date_year",              "date_part",   "feature"),
    ("arrival_date_month",             "date_part",   "feature"),
    ("arrival_date_week_number",       "date_part",   "feature"),
    ("arrival_date_day_of_month",      "date_part",   "feature"),
    ("stays_in_weekend_nights",        "count",       "feature"),
    ("stays_in_week_nights",           "count",       "feature"),
    ("adults",                         "count",       "feature"),
    ("children",                       "count",       "feature"),
    ("babies",                         "count",       "feature"),
    ("meal",                           "categorical", "feature"),
    ("country",                        "categorical", "feature"),
    ("market_segment",                 "categorical", "feature"),
    ("distribution_channel",           "categorical", "feature"),
    ("is_repeated_guest",              "binary",      "feature"),
    ("previous_cancellations",         "count",       "feature"),
    ("previous_bookings_not_canceled", "count",       "feature"),
    ("reserved_room_type",             "categorical", "feature"),
    ("assigned_room_type",             "categorical", "forbidden"),
    ("booking_changes",                "count",       "forbidden"),
    ("deposit_type",                   "categorical", "forbidden"),
    ("agent",                          "id_code",     "feature"),
    ("company",                        "id_code",     "feature"),
    ("days_in_waiting_list",           "count_days",  "feature"),
    ("customer_type",                  "categorical", "feature"),
    ("adr",                            "continuous",  "target_reg"),
    ("required_car_parking_spaces",    "count",       "forbidden"),
    ("total_of_special_requests",      "count",       "feature"),
    ("reservation_status",             "categorical", "outcome"),
    ("reservation_status_date",        "date",        "outcome"),
], columns=["column", "analytic_type", "role"])

check(set(DATA_DICTIONARY["column"]) == set(raw.columns),
      "The dictionary covers all 32 columns.", "Dictionary and file do not match.")
DATA_DICTIONARY["role"].value_counts()

✅ The dictionary covers all 32 columns.


role
feature       24
forbidden      4
outcome        2
target_cls     1
target_reg     1
Name: count, dtype: int64

**Copia con tipos (`df`).** `raw` no se toca. En `df` pasamos a números y fechas con reglas fijas, que no
aprenden nada de los datos.

In [4]:
NUMERIC = DATA_DICTIONARY.query("analytic_type in ['binary', 'count', 'count_days', 'continuous']")["column"].tolist()

df = raw.copy()
df[NUMERIC] = df[NUMERIC].apply(pd.to_numeric, errors="coerce")
df["arrival_date"] = pd.to_datetime(
    raw["arrival_date_year"] + "-" + raw["arrival_date_month"] + "-" + raw["arrival_date_day_of_month"],
    format="%Y-%B-%d", errors="coerce",
)
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df[["is_canceled", "adr", "lead_time", "total_nights", "arrival_date"]].describe()

,is_canceled,adr,lead_time,total_nights,arrival_date
count,119390.000000,119390.000000,119390.000000,119390.000000,119390
mean,0.370416,101.831122,104.011416,3.427900,2016-08-28 16:39:45.727447808
min,0.000000,-6.380000,0.000000,0.000000,2015-07-01 00:00:00
25%,0.000000,69.290000,18.000000,2.000000,2016-03-13 00:00:00
50%,0.000000,94.575000,69.000000,3.000000,2016-09-06 00:00:00
75%,1.000000,126.000000,160.000000,4.000000,2017-03-18 00:00:00
max,1.000000,5400.000000,737.000000,69.000000,2017-08-31 00:00:00
std,0.482918,50.535790,106.863097,2.557439,NaN


## 3 · ¿Hay duplicados?

No hay ID de reserva (se borró para anonimizar). Sin ID no sabemos si dos filas iguales son **un error** o
**dos reservas parecidas**. Aquí solo medimos; se decide en el 02.

In [5]:
dup_all = raw.duplicated(keep=False)  # every row that has a twin
dup_extra = raw.duplicated()          # only the surplus copies

pd.Series({
    "surplus exact copies (%)": pct(dup_extra),
    "canceled among duplicated (%)": pct(df.loc[dup_all, "is_canceled"] == 1),
    "canceled among the rest (%)": pct(df.loc[~dup_all, "is_canceled"] == 1),
}, name="value").to_frame()

,value
surplus exact copies (%),26.8
canceled among duplicated (%),58.4
canceled among the rest (%),26.2


**Por qué importa:** los duplicados se cancelan más del doble. Si una copia cae en *train* y su gemela en
*test*, el modelo «ya ha visto» la respuesta.

## 4 · ¿Qué columnas no se pueden usar?

El modelo predice **al reservar**. Si una columna se rellena después, usarla es hacer trampa (**fuga**).

**Limitación del dataset (paper):** los valores no son los del momento de la reserva, sino los del **día
anterior a la llegada** (*«the data point time for each observation was defined as the day prior to each
booking's arrival»*). Lo que se sabía al reservar no se puede recuperar; solo podemos elegir qué columnas usar.

- **Prohibidas:** se conocen después de reservar o son el propio desenlace.
- **Dudosas:** existen al reservar, pero el valor guardado es el del día antes de llegar y puede haber
  cambiado. El paper avisa de que `adults`, `children`, las `stays_*`, `meal` y `country` se distribuyen
  distinto entre canceladas y no canceladas → se probará con y sin ellas, **por tarea** (la lista no es
  la misma para clasificación y regresión).

In [6]:
FORBIDDEN = [
    "required_car_parking_spaces",  # data: no canceled booking has any → most likely recorded on arrival
    "booking_changes",              # paper: counts changes up to check-in or cancellation
    "assigned_room_type",           # paper: may differ from the reserved one; data: most likely set at check-in
    "deposit_type",                 # paper: computed from the payments made before arrival or cancellation
    "reservation_status",           # it is the outcome (Canceled or No-Show → is_canceled = 1)
    "reservation_status_date",      # date of the outcome
]
# Exist at booking, but the stored value is the one of the day before arrival (paper).
# The list depends on the task: a column that may give away the outcome (cancel or not) can still be a
# legitimate input for the price, because the quote is made for those nights, guests and meal plan.
UNDER_REVIEW = {
    "classification": [
        "stays_in_*_nights",           # paper: differ between canceled and non-canceled bookings
        "adults / children / babies",  # paper: adults and children differ between canceled and non-canceled
        "meal",                        # paper: differs between canceled and non-canceled bookings
        "days_in_waiting_list",        # paper: confirmation date − entry date, unknown until confirmed
        "total_of_special_requests",   # requests can be added up to arrival
        "country",                     # paper: nationality often unknown until check-in
        "adr (as an input)",           # paper: lodging transactions ÷ nights, depends on the final stay
    ],
    "regression": [  # nights, guests and meal define what is priced → allowed (limitation: day-before-arrival value)
        "days_in_waiting_list",
        "total_of_special_requests",
        "country",
    ],
}
check(set(FORBIDDEN) == set(DATA_DICTIONARY.query("role in ['forbidden', 'outcome']")["column"]),
      "FORBIDDEN matches the roles of the data dictionary.", "FORBIDDEN and the data dictionary disagree.")

# The target: cancellations and no-shows are both is_canceled = 1
outcome = raw["reservation_status"].isin(["Canceled", "No-Show"]).astype(int).astype(str)
check(outcome.eq(raw["is_canceled"]).all(),
      f"is_canceled = 1 ⇔ Canceled or No-Show ({raw['reservation_status'].eq('No-Show').sum():,} no-shows).",
      "is_canceled does not match reservation_status.")

# Evidence: if a column is filled in after booking, canceled bookings (which never arrive) look very different
canc = df["is_canceled"] == 1
room_changed = raw["assigned_room_type"] != raw["reserved_room_type"]
pd.DataFrame({
    "arrived": [pct(df.loc[~canc, "required_car_parking_spaces"] > 0), pct(room_changed[~canc]),
                pct(df.loc[~canc, "booking_changes"] > 0), pct(raw.loc[~canc, "deposit_type"] == "Non Refund")],
    "canceled": [pct(df.loc[canc, "required_car_parking_spaces"] > 0), pct(room_changed[canc]),
                 pct(df.loc[canc, "booking_changes"] > 0), pct(raw.loc[canc, "deposit_type"] == "Non Refund")],
}, index=["with parking (%)", "assigned room ≠ reserved (%)", "with booking changes (%)", "Non Refund (%)"])

✅ FORBIDDEN matches the roles of the data dictionary.
✅ is_canceled = 1 ⇔ Canceled or No-Show (1,207 no-shows).


,arrived,canceled
with parking (%),9.9,0.0
assigned room ≠ reserved (%),18.8,1.8
with booking changes (%),20.3,6.4
Non Refund (%),0.1,32.8


**Lectura**

- **Parking:** **ninguna** reserva cancelada lo tiene. El paper no dice cuándo se registra; los datos apuntan
  a que se anota al llegar → se prohíbe por los datos, no por el paper.
- **Habitación asignada:** el paper dice que puede diferir de la reservada por *overbooking* o a petición del
  cliente. Cambia en el 18,8 % de las reservas que llegan y solo en el 1,8 % de las canceladas → todo apunta a
  que se asigna en el *check-in*.
- **Cambios en la reserva:** el paper los cuenta hasta el *check-in* o la cancelación. El 20,3 % de las que
  llegan tiene alguno, frente al 6,4 % de las canceladas: una reserva que acaba de entrar siempre tendría 0.
- **Depósito:** el paper lo calcula con los pagos hechos **antes de la llegada o de la cancelación**, no al
  reservar. Por eso `Non Refund` casi solo aparece en canceladas (32,8 % frente a 0,1 %).
- `reservation_status` es el desenlace: `Canceled` y `No-Show` son `is_canceled = 1`. El objetivo es
  «cancela **o no se presenta**».

Las **dudosas** no se prohíben porque el dato existe al reservar; lo que no sabemos es cuánto cambió hasta la
víspera de la llegada. El comentario de cada una en el código dice por qué.

**La lista depende de la tarea.** Noches, huéspedes y `meal` son dudosas en la **clasificación** porque se
distribuyen distinto entre canceladas y no canceladas: pueden delatar el desenlace. En la **regresión** son
justo lo que se cotiza (el precio por noche se calcula para esas noches, esas personas y ese régimen), así
que se admiten como entrada, con la limitación de que el valor guardado es el de la víspera. El 03 medirá
cada modelo con y sin sus dudosas.

## 5 · ¿Cuánta calidad tienen los datos?

Una pregunta por dimensión: ¿**falta** el dato? ¿el valor es **posible**?

In [7]:
ADR_MAX = 1_000  # domain limit (the 2nd highest adr is 510)
guests = df["adults"] + df["children"].fillna(0) + df["babies"]

quality = pd.DataFrame([
    ("completeness", "adr present (%)",                pct(df["adr"].notna())),
    ("completeness", "country present (%)",            pct(raw["country"] != "NULL")),
    ("validity",     "is_canceled in {0, 1} (%)",      pct(df["is_canceled"].isin([0, 1]))),
    ("validity",     "arrival 2015-07 to 2017-08 (%)", pct(df["arrival_date"].between("2015-07-01", "2017-08-31"))),
    ("validity",     "rows with adr < 0",              int((df["adr"] < 0).sum())),
    ("validity",     f"rows with adr > {ADR_MAX}",     int((df["adr"] > ADR_MAX).sum())),
    ("validity",     "rows without guests",            int((guests == 0).sum())),
    ("validity",     "rows with 0 nights",             int((df["total_nights"] == 0).sum())),
], columns=["dimension", "metric", "value"])
quality

,dimension,metric,value
0,completeness,adr present (%),100.0
1,completeness,country present (%),99.6
2,validity,"is_canceled in {0, 1} (%)",100.0
3,validity,arrival 2015-07 to 2017-08 (%),100.0
4,validity,rows with adr < 0,1.0
5,validity,rows with adr > 1000,1.0
6,validity,rows without guests,180.0
7,validity,rows with 0 nights,715.0


**Lectura**

- Los **objetivos** están completos y son válidos.
- **182 filas imposibles** (`adr` negativo, `adr` de 5 400, sin huéspedes) → cuarentena en el 02.
- **715 reservas de 0 noches:** `adr` es «por noche», sin noches no tiene sentido → fuera de la regresión.

## 6 · ¿A quién representan?

In [8]:
pd.Series({
    "City Hotel (%)": pct(raw["hotel"] == "City Hotel"),
    "guests from Portugal (%)": pct(raw["country"] == "PRT"),
    "months covered": df["arrival_date"].dt.to_period("M").nunique(),
}, name="value").to_frame()

,value
City Hotel (%),66.4
guests from Portugal (%),40.7
months covered,26.0


**Lectura:** dos hoteles portugueses entre 2015 y 2017 → el modelo **no es general**. Son 26 meses: julio y
agosto aparecen tres veces, así que las medias por mes tiran hacia el verano.

## 7 · ¿Pasa el lote al notebook 02?

Cada control es **PASS** o **FAIL**. La regla:

| Situación | Decisión |
|---|---|
| Falla un control **crítico** que afecta a todo el `dataset` | `QUARANTINE`: se para |
| Falla otra cosa (filas o columnas concretas) | `CONTINUE_WITH_CONDITIONS` |
| Todo pasa | `PUBLISH` |

In [9]:
q = quality.set_index("metric")["value"]
impossible = (df["adr"] < 0) | (df["adr"] > ADR_MAX) | (guests == 0)

scorecard = pd.DataFrame([
    # control,           scope,      passed,                                              severity,   on failure
    ("paper_counts",      "dataset",  paper_counts_ok,                                     "critical", "investigate"),
    ("source_hash",       "dataset",  raw_sha256 == SOURCE["expected_sha256"],             "critical", "download again"),
    ("rows_and_columns",  "dataset",  raw.shape == (119_390, 32),                          "critical", "investigate"),
    ("targets_valid",     "dataset",  q["is_canceled in {0, 1} (%)"] == 100 and q["adr present (%)"] == 100,
                                                                                           "critical", "investigate"),
    ("period_covered",    "dataset",  q["arrival 2015-07 to 2017-08 (%)"] == 100,          "critical", "investigate"),
    ("exact_copies_<5%",  "dataset",  pct(dup_extra) < 5,                                  "warning",  "define a duplicates policy"),  # 5 %: see note below
    ("impossible_rows",   "row",      not impossible.any(),                                "critical", "quarantine"),
    ("zero_nights",       "row",      q["rows with 0 nights"] == 0,                        "warning",  "exclude from the regression"),
    ("forbidden_columns", "features", len(FORBIDDEN) == 0,                                 "critical", "exclude from the features"),
], columns=["control", "scope", "passed", "severity", "on_failure"])
scorecard["passed"] = scorecard["passed"].astype(bool)

blocking = scorecard.query("not passed and severity == 'critical' and scope == 'dataset'")
batch_decision = "QUARANTINE" if len(blocking) else ("PUBLISH" if scorecard["passed"].all() else "CONTINUE_WITH_CONDITIONS")
print("Batch decision:", batch_decision)
scorecard

Batch decision: CONTINUE_WITH_CONDITIONS


,control,scope,passed,severity,on_failure
0,paper_counts,dataset,True,critical,investigate
1,source_hash,dataset,True,critical,download again
2,rows_and_columns,dataset,True,critical,investigate
3,targets_valid,dataset,True,critical,investigate
4,period_covered,dataset,True,critical,investigate
5,exact_copies_<5%,dataset,False,warning,define a duplicates policy
6,impossible_rows,row,False,critical,quarantine
7,zero_nights,row,False,warning,exclude from the regression
8,forbidden_columns,features,False,critical,exclude from the features


- `forbidden_columns` falla **a propósito**: las columnas están en el fichero. Se cumple cuando el 03 las deje
  fuera del modelo.
- `exact_copies_<5%`: el 5 % es una convención, no sale de los datos. Por eso el control es solo un **aviso**
  (no para el lote) y su efecto es obligar a decidir qué hacer con los duplicados. Con un 26,8 % de copias, el
  aviso salta con cualquier umbral razonable.

## 8 · Guardar lo que necesita el notebook 02

- **Contrato** (`data_contract.json`): qué columnas hay, qué significan y cuáles están prohibidas.
- **Resumen** (`quality_summary.json`): la decisión, las tareas pendientes y con qué se ejecutó.

In [10]:
SENTINELS = {  # text values with a special meaning; "source" says who defines it
    "agent": {"value": "NULL", "meaning": "not applicable (booking without an agency)", "source": "paper"},
    "company": {"value": "NULL", "meaning": "not applicable (booking without a company)", "source": "paper"},
    "meal": {"value": "Undefined", "meaning": "same as SC (no meal plan)", "source": "paper"},
    # The paper says the PMS has no missing data: reading these as "missing" is our decision
    "children": {"value": "NA", "meaning": "missing", "source": "our decision"},
    "country": {"value": "NULL", "meaning": "missing", "source": "our decision"},
    "market_segment": {"value": "Undefined", "meaning": "missing", "source": "our decision"},
    "distribution_channel": {"value": "Undefined", "meaning": "missing", "source": "our decision"},
}
schema = {r.column: {"type": r.analytic_type, "role": r.role} for r in DATA_DICTIONARY.itertuples()}
for col, sentinel in SENTINELS.items():
    schema[col]["sentinel"] = sentinel
schema["adr"].update({"min": 0, "max": ADR_MAX, "unit": "€/night (currency not stated in the paper)"})

data_contract = {
    "identity": {"dataset": SOURCE["name"], "version": "1.1.0", "doi": SOURCE["doi"],
                 "license": SOURCE["license"], "sha256": raw_sha256},
    "purpose": {"targets": {"classification": "is_canceled", "regression": "adr"},
                "prediction_time": "when the booking is registered",
                "data_snapshot": "day before arrival (paper): stored values may differ from those at booking",
                "forbidden_features": FORBIDDEN, "features_under_review": UNDER_REVIEW},
    "schema": schema,
    "quality": scorecard[["control", "scope", "severity", "on_failure"]].to_dict("records"),
}

quality_summary = {
    "dataset": SOURCE["name"],
    "decision": batch_decision,
    "rows_to_quarantine": int(impossible.sum()),
    "rows_excluded_from_regression": int((df["total_nights"] == 0).sum()),
    "pending_decisions_for_02": [
        "exact duplicates: drop, group within the split or weight",
        "adr = 0 with nights > 0 (mostly Complementary): exclude from the regression or not",
        "CN versus CHN: unify or not (compare both populations)",
        "sentinels: NULL in agent/company = 'none'; NULL in country = 'unknown'; Undefined in meal = SC",
    ],
    "executed_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "numpy": np.__version__, "seed": SEED},
}

(OUT_DIR / "data_contract.json").write_text(json.dumps(data_contract, indent=2, ensure_ascii=False), encoding="utf-8")
(OUT_DIR / "quality_summary.json").write_text(json.dumps(quality_summary, indent=2, ensure_ascii=False), encoding="utf-8")
scorecard.to_csv(OUT_DIR / "scorecard.csv", index=False)
print("Saved in", OUT_DIR.relative_to(ROOT).as_posix(), "→ data_contract.json, quality_summary.json, scorecard.csv")
quality_summary

Saved in outputs/01-data-collection-and-quality → data_contract.json, quality_summary.json, scorecard.csv


{'dataset': 'hotel-booking-demand',
 'decision': 'CONTINUE_WITH_CONDITIONS',
 'rows_to_quarantine': 182,
 'rows_excluded_from_regression': 715,
 'pending_decisions_for_02': ['exact duplicates: drop, group within the split or weight',
  'adr = 0 with nights > 0 (mostly Complementary): exclude from the regression or not',
  'CN versus CHN: unify or not (compare both populations)',
  "sentinels: NULL in agent/company = 'none'; NULL in country = 'unknown'; Undefined in meal = SC"],
 'executed_at': '2026-10-20T11:48:14+00:00',
 'environment': {'python': '3.14.4',
  'pandas': '2.3.3',
  'numpy': '2.3.5',
  'seed': 42}}

**Siguiente:** `02-eda-and-cleaning.ipynb` lee estos dos JSON, explora los datos y aplica las decisiones
pendientes.